# MarkItDown Document to Markdown Converter

This notebook converts PDF, DOCX, and other document formats to Markdown with Microsoft `markitdown`. For PDFs, the default flow uses MarkItDown when a text layer exists and offline Tesseract OCR when the PDF appears scanned.

In [ ]:
# ===============================================================
# CONFIG - EDIT ONLY THIS CELL
# ===============================================================
from pathlib import Path

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    files = None
    IN_COLAB = False

# Choose input mode:
# - "upload": choose supported file(s) directly; outputs auto-download in Colab.
# - "drive": read supported file(s) from DRIVE_INPUT_DIR; outputs are saved to DRIVE_OUTPUT_DIR.
INPUT_MODE = "upload"
AUTO_DOWNLOAD = True

# Local Jupyter input folder used outside Colab when INPUT_MODE="upload" cannot open a Colab chooser.
LOCAL_INPUT_DIR = Path("./input_documents")

# Google Drive folder mode. Change these to your real Drive paths, then use INPUT_MODE="drive".
DRIVE_INPUT_DIR = Path("/content/drive/MyDrive/input_documents")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/output_markdown")
RECURSIVE_DRIVE_SEARCH = True
PRESERVE_DRIVE_SUBFOLDERS = True

CONVERSION_CONFIG = {
    # pdf_method notes:
    # - pdf_method="auto" is the default: use MarkItDown for text PDFs and Tesseract for scanned PDFs.
    # - Use pdf_method="offline_ocr" to force Tesseract OCR for every PDF page.
    # - Enable markitdown-ocr only when you want vision-model OCR for scans or embedded images.
    # Options: "auto", "markitdown", "offline_ocr".
    "pdf_method": "auto",
    "scan_detection_pages": 3,
    "scan_min_chars_per_page": 30,

    # Offline OCR settings. Vietnamese + English is the default OCR language.
    "offline_ocr_language": "vie+eng",
    "offline_ocr_dpi": 220,
    "offline_ocr_max_pages": None,

    # Keep plugins off by default for predictable local/Colab runs.
    "enable_plugins": False,

    # Optional only: set True after installing markitdown-ocr and configuring an OpenAI-compatible client.
    "use_ocr_plugin": False,
    "llm_model": "gpt-4o",
    "llm_prompt": None,

    # Optional Azure Document Intelligence endpoint for PDF/layout-heavy cases.
    "docintel_endpoint": None,

    "min_output_chars": 20,
    "output_dir": "/content" if IN_COLAB else "./converted_markdown",
}

SUPPORTED_INPUT_SUFFIXES = {
    ".pdf", ".docx", ".pptx", ".xlsx", ".xls",
    ".html", ".htm", ".txt", ".csv", ".json", ".xml",
    ".zip", ".epub", ".jpg", ".jpeg", ".png", ".wav", ".mp3",
}


## Step 1: Install Required Libraries

In [ ]:
# Full MarkItDown support for every suffix listed in SUPPORTED_INPUT_SUFFIXES, plus offline OCR helpers.
%pip install -q "markitdown[all]" pymupdf pillow pytesseract

import platform
import shutil
import subprocess


def install_tesseract_language_data_if_needed() -> None:
    if platform.system() != "Linux" or shutil.which("apt-get") is None:
        return

    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "tesseract-ocr", "tesseract-ocr-vie"], check=True)


def installed_tesseract_languages() -> set[str]:
    result = subprocess.run(
        ["tesseract", "--list-langs"],
        capture_output=True,
        text=True,
        check=True,
    )
    return {
        line.strip()
        for line in result.stdout.splitlines()
        if line.strip() and not line.lower().startswith("list of available")
    }


if shutil.which("tesseract") is None:
    install_tesseract_language_data_if_needed()
else:
    print("Tesseract is already installed.")

if shutil.which("tesseract") is None:
    print("Tesseract is not installed. Install it locally, for example: brew install tesseract tesseract-lang")
else:
    languages = installed_tesseract_languages()
    if "vie" not in languages:
        install_tesseract_language_data_if_needed()
        languages = installed_tesseract_languages()
    if "vie" not in languages:
        print("Vietnamese Tesseract language data is missing. Install it locally, for example: brew install tesseract-lang")
    else:
        print("Vietnamese Tesseract language data is available.")

# Optional: OCR plugin for image-heavy/scanned documents via an OpenAI-compatible vision model.
# %pip install -q markitdown-ocr openai


## Step 2: Import Libraries

In [ ]:
import os
import uuid
from pathlib import Path
from typing import Iterable, Optional

import fitz
import pytesseract
from markitdown import MarkItDown
from PIL import Image

input_files = []
output_dir = Path(CONVERSION_CONFIG["output_dir"])
converted_files = []


## Step 3: Upload or Select Input Files (Optional)

In [ ]:
def iter_supported_files(input_dir: Path, recursive: bool = True) -> list[Path]:
    pattern = "**/*" if recursive else "*"
    return [
        path for path in sorted(input_dir.glob(pattern))
        if path.is_file() and path.suffix.lower() in SUPPORTED_INPUT_SUFFIXES
    ]


def output_path_for(input_file: Path, output_root: Path, input_root: Path | None = None) -> Path:
    if INPUT_MODE == "drive" and PRESERVE_DRIVE_SUBFOLDERS and input_root is not None:
        try:
            relative_parent = input_file.parent.relative_to(input_root)
            target_dir = output_root / relative_parent
        except ValueError:
            target_dir = output_root
    else:
        target_dir = output_root
    target_dir.mkdir(parents=True, exist_ok=True)
    return target_dir / f"{input_file.stem}.md"


def validate_unique_output_paths(input_paths: list[Path], output_root: Path, input_root: Path | None = None) -> None:
    seen = {}
    for input_path in input_paths:
        output_path = output_path_for(input_path, output_root, input_root)
        key = str(output_path.resolve() if output_path.exists() else output_path.absolute())
        if key in seen:
            raise ValueError(
                f"Duplicate output path would be created for {seen[key]} and {input_path}: {output_path}. "
                "Set PRESERVE_DRIVE_SUBFOLDERS=True or rename one input file."
            )
        seen[key] = input_path


def upload_or_select_files() -> list[Path]:
    if INPUT_MODE == "drive":
        print("INPUT_MODE='drive': direct upload/local selection skipped; Drive folder will be mounted later.")
        return []

    if INPUT_MODE != "upload":
        raise ValueError("INPUT_MODE must be either 'upload' or 'drive'.")

    if IN_COLAB:
        print("Choose supported file(s) from your computer:")
        uploaded = files.upload()
        selected_files = []
        for filename in uploaded.keys():
            file_path = Path("/content") / filename
            if file_path.suffix.lower() not in SUPPORTED_INPUT_SUFFIXES:
                raise ValueError(f"Unsupported or untested file type: {file_path.suffix}")
            selected_files.append(file_path)
        if not selected_files:
            raise ValueError("No supported file was uploaded.")
        return selected_files

    input_folder = LOCAL_INPUT_DIR
    if not input_folder.exists():
        raise FileNotFoundError(
            "Not running in Colab. Put files in LOCAL_INPUT_DIR, or run this notebook in Google Colab for direct upload."
        )
    selected_files = iter_supported_files(input_folder, RECURSIVE_DRIVE_SEARCH)
    if not selected_files:
        raise FileNotFoundError(f"No supported files found in: {input_folder}")
    return selected_files


input_files = upload_or_select_files()
output_dir = Path(CONVERSION_CONFIG["output_dir"])
input_root = None
output_dir.mkdir(parents=True, exist_ok=True)

if input_files:
    print(f"Selected {len(input_files)} file(s):")
    for file_path in input_files:
        print(f"- {file_path.name} ({file_path.stat().st_size / 1024 / 1024:.2f} MB)")


## Step 4: Mount Google Drive Folder Mode (Optional)

In [ ]:
if INPUT_MODE == "drive":
    if not IN_COLAB:
        raise RuntimeError("Google Drive folder mode is available in Google Colab only.")

    from google.colab import drive

    drive.mount("/content/drive")

    DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    input_root = DRIVE_INPUT_DIR
    input_files = iter_supported_files(DRIVE_INPUT_DIR, RECURSIVE_DRIVE_SEARCH)
    output_dir = DRIVE_OUTPUT_DIR
    CONVERSION_CONFIG["output_dir"] = str(DRIVE_OUTPUT_DIR)

    if not input_files:
        raise FileNotFoundError(f"No supported files found in: {DRIVE_INPUT_DIR}")

    print("Google Drive mounted successfully.")
    print(f"Input folder: {DRIVE_INPUT_DIR}")
    print(f"Output folder: {DRIVE_OUTPUT_DIR}")
    print(f"Found {len(input_files)} supported file(s).")
else:
    print("INPUT_MODE='upload': Google Drive folder mode skipped.")


## Step 5: Define Conversion Helpers

In [ ]:
def build_markitdown(config: dict = CONVERSION_CONFIG) -> MarkItDown:
    kwargs = {
        "enable_plugins": bool(config.get("enable_plugins") or config.get("use_ocr_plugin")),
    }

    if config.get("docintel_endpoint"):
        kwargs["docintel_endpoint"] = config["docintel_endpoint"]

    if config.get("use_ocr_plugin"):
        from openai import OpenAI

        kwargs["llm_client"] = OpenAI()
        kwargs["llm_model"] = config.get("llm_model", "gpt-4o")
        if config.get("llm_prompt"):
            kwargs["llm_prompt"] = config["llm_prompt"]

    return MarkItDown(**kwargs)


def default_output_path(input_file: Path, output_dir: Optional[str] = None) -> Path:
    base_dir = Path(output_dir or CONVERSION_CONFIG["output_dir"])
    base_dir.mkdir(parents=True, exist_ok=True)
    return base_dir / f"{input_file.stem}.md"


def is_scanned_or_image_only_pdf(input_file: Path, config: dict = CONVERSION_CONFIG) -> bool:
    sample_pages = config.get("scan_detection_pages", 3)
    min_chars_per_page = config.get("scan_min_chars_per_page", 30)

    with fitz.open(input_file) as doc:
        pages_to_check = min(len(doc), sample_pages)
        if pages_to_check == 0:
            return True

        extracted_chars = 0
        for page_index in range(pages_to_check):
            extracted_chars += len(doc[page_index].get_text("text").strip())

    avg_chars = extracted_chars / pages_to_check
    print(f"PDF text-layer check: {avg_chars:.0f} extractable chars/page")
    return avg_chars < min_chars_per_page


def ocr_pdf_offline(input_file: Path, config: dict = CONVERSION_CONFIG) -> str:
    dpi = int(config.get("offline_ocr_dpi", 220))
    language = config.get("offline_ocr_language", "vie+eng")
    max_pages = config.get("offline_ocr_max_pages")
    zoom = dpi / 72
    matrix = fitz.Matrix(zoom, zoom)
    md_pages = []

    with fitz.open(input_file) as doc:
        total_pages = len(doc)
        pages_to_process = total_pages if max_pages is None else min(total_pages, int(max_pages))
        print(f"Running offline OCR with Tesseract: {pages_to_process}/{total_pages} pages, lang={language}, dpi={dpi}")

        for page_index in range(pages_to_process):
            page = doc[page_index]
            pix = page.get_pixmap(matrix=matrix, alpha=False)
            image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
            text = pytesseract.image_to_string(image, lang=language).strip()
            md_pages.append(f"## Page {page_index + 1}\n\n{text}\n")

    return "\n\n".join(md_pages).strip() + "\n"


def convert_with_markitdown(input_file: Path, config: dict = CONVERSION_CONFIG) -> str:
    md = build_markitdown(config)
    result = md.convert(str(input_file))
    return result.text_content or ""


def convert_pdf_to_markdown(input_file: Path, config: dict = CONVERSION_CONFIG) -> str:
    method = config.get("pdf_method", "auto")

    if method == "offline_ocr":
        return ocr_pdf_offline(input_file, config)

    if method == "markitdown":
        return convert_with_markitdown(input_file, config)

    if method != "auto":
        raise ValueError(f"Unsupported pdf_method: {method}")

    if is_scanned_or_image_only_pdf(input_file, config):
        return ocr_pdf_offline(input_file, config)

    return convert_with_markitdown(input_file, config)


def convert_file_to_markdown(
    input_file: Path,
    output_path: Optional[Path] = None,
    config: dict = CONVERSION_CONFIG,
) -> str:
    input_file = Path(input_file)
    output_path = Path(output_path) if output_path else default_output_path(input_file)

    if not input_file.exists():
        raise FileNotFoundError(f"Input file does not exist: {input_file}")

    if input_file.suffix.lower() == ".pdf":
        md_text = convert_pdf_to_markdown(input_file, config)
    else:
        md_text = convert_with_markitdown(input_file, config)

    if len(md_text.strip()) < config.get("min_output_chars", 20):
        raise RuntimeError(
            "Conversion produced little or no text. For PDF scans, check Tesseract language data, "
            "increase offline_ocr_dpi, or try the optional markitdown-ocr/Azure paths."
        )

    temp_output_path = output_path.with_name(f".{output_path.stem}.{uuid.uuid4().hex}.tmp{output_path.suffix}")
    try:
        temp_output_path.write_text(md_text, encoding="utf-8")
        if not temp_output_path.exists() or temp_output_path.stat().st_size == 0:
            raise RuntimeError(f"Conversion failed or produced an empty file: {temp_output_path}")
        temp_output_path.replace(output_path)
    finally:
        if temp_output_path.exists():
            temp_output_path.unlink()

    print(f"Saved: {output_path}")
    print(f"Content length: {len(md_text):,} characters")
    return md_text


def iter_supported_files_for_batch(folder: Path) -> Iterable[Path]:
    folder = Path(folder)
    for path in sorted(folder.rglob("*")):
        if path.is_file() and path.suffix.lower() in SUPPORTED_INPUT_SUFFIXES:
            yield path


## Step 6: Convert Files

In [ ]:
if not input_files:
    raise ValueError("No input files. Check INPUT_MODE and the configured input source in the Configuration cell.")

validate_unique_output_paths(input_files, Path(output_dir), input_root)
converted_files = []

for input_path in input_files:
    output_path = output_path_for(input_path, Path(output_dir), input_root)
    md_content = convert_file_to_markdown(input_path, output_path, CONVERSION_CONFIG)

    if len(md_content.strip()) < CONVERSION_CONFIG.get("min_output_chars", 20):
        raise RuntimeError(f"Conversion produced little or no text for {input_path.name}. Check OCR/config or document quality.")
    if not output_path.exists() or output_path.stat().st_size == 0:
        raise RuntimeError(f"Conversion failed or produced an empty file for {input_path.name}.")

    converted_files.append(output_path)
    print(f"Converted: {input_path.name} -> {output_path}")

print()
print(f"Conversion completed for {len(converted_files)} file(s).")
print(f"Output folder: {output_dir}")


## Step 7: Download Converted File (Colab)

In [ ]:
if not converted_files:
    raise FileNotFoundError("No converted files found. Run the conversion cell first.")

if INPUT_MODE == "drive":
    print(f"Output files are saved in Google Drive: {Path(output_dir).resolve()}")
elif IN_COLAB and AUTO_DOWNLOAD:
    for output_path in converted_files:
        if not output_path.exists() or output_path.stat().st_size == 0:
            raise FileNotFoundError(f"Output file is missing or empty: {output_path}")
        files.download(str(output_path))
    print(f"Downloaded {len(converted_files)} file(s).")
else:
    print(f"AUTO_DOWNLOAD=False or not running in Colab. Output files are ready at: {Path(output_dir).resolve()}")


## Step 8: Batch Convert a Folder (Optional)

In [ ]:
# Change these paths, then uncomment to convert all supported files in a folder.
# input_folder = Path("/content/input_documents" if IN_COLAB else "./input_documents")
# batch_output_dir = Path(CONVERSION_CONFIG["output_dir"])
# batch_results = []

# for source_file in iter_supported_files_for_batch(input_folder):
#     try:
#         target_file = batch_output_dir / f"{source_file.stem}.md"
#         text = convert_file_to_markdown(source_file, target_file, CONVERSION_CONFIG)
#         batch_results.append((source_file.name, target_file.name, len(text), "ok"))
#     except Exception as exc:
#         batch_results.append((source_file.name, "", 0, f"error: {exc}"))

# batch_results


## Step 9: Force Offline OCR Mode (Optional)

In [ ]:
# Use this when you want OCR for every PDF page, even if the file has a text layer.
# This is fully offline after Tesseract is installed.

# CONVERSION_CONFIG["pdf_method"] = "offline_ocr"
# CONVERSION_CONFIG["offline_ocr_language"] = "vie+eng"
# CONVERSION_CONFIG["offline_ocr_dpi"] = 300
# CONVERSION_CONFIG["offline_ocr_max_pages"] = None

# offline_ocr_output_path = default_output_path(input_path).with_name(f"{input_path.stem}_offline_ocr.md")
# md_content_offline_ocr = convert_file_to_markdown(input_path, offline_ocr_output_path, CONVERSION_CONFIG)
# print(md_content_offline_ocr[:3000])


## Step 10: MarkItDown OCR Plugin / Vision OCR (Optional)

In [ ]:
# Optional cloud/LLM OCR path. Use only when offline Tesseract quality is not enough.
# Requires OPENAI_API_KEY in the environment and may incur API cost.

# %pip install -q markitdown-ocr openai
# os.environ["OPENAI_API_KEY"] = "..."  # Avoid hard-coding keys in shared notebooks.
# CONVERSION_CONFIG["pdf_method"] = "markitdown"
# CONVERSION_CONFIG["use_ocr_plugin"] = True
# CONVERSION_CONFIG["enable_plugins"] = True
# CONVERSION_CONFIG["llm_model"] = "gpt-4o"

# vision_ocr_output_path = default_output_path(input_path).with_name(f"{input_path.stem}_vision_ocr.md")
# md_content_vision_ocr = convert_file_to_markdown(input_path, vision_ocr_output_path, CONVERSION_CONFIG)
# print(md_content_vision_ocr[:3000])


## Step 11: Azure Document Intelligence Mode (Optional)

In [ ]:
# Use this for layout-heavy PDFs if you have an Azure Document Intelligence endpoint.

# CONVERSION_CONFIG["docintel_endpoint"] = "https://<your-resource>.cognitiveservices.azure.com/"
# docintel_output_path = default_output_path(input_path).with_name(f"{input_path.stem}_docintel.md")
# md_content_docintel = convert_file_to_markdown(input_path, docintel_output_path, CONVERSION_CONFIG)
# print(md_content_docintel[:3000])
